In [ ]:
# file: q1_stft_mel.py
import numpy as np
import librosa
import matplotlib.pyplot as plt
import scipy.signal

# --- parameters ---
wav_path = "sample_audio.wav"   # download sample or any short wav
sr_target = 22050               # target sampling rate
n_fft = 2048
hop_length = 512
win_length = n_fft
n_mels = 128
fmin = 0.0
fmax = None  # set to sr/2 later

# --- load ---
y, sr = librosa.load(wav_path, sr=sr_target, mono=True)
if fmax is None:
    fmax = sr / 2.

# --- framing + window ---
def stft_from_scratch(y, n_fft=2048, hop_length=512, win_length=None, window='hann'):
    if win_length is None:
        win_length = n_fft
    w = scipy.signal.get_window(window, win_length, fftbins=True)
    # pad signal to center windows
    pad = int(n_fft // 2)
    y = np.pad(y, pad_width=pad, mode='reflect')
    # number of frames
    n_frames = 1 + (len(y) - n_fft) // hop_length
    stft = np.empty((n_fft//2 + 1, n_frames), dtype=np.complex64)
    for i in range(n_frames):
        start = i * hop_length
        frame = y[start:start + n_fft]
        # apply window (if win_length < n_fft, center it)
        if win_length != n_fft:
            wfull = np.zeros(n_fft)
            startw = (n_fft - win_length)//2
            wfull[startw:startw+win_length] = w
            frame = frame * wfull
        else:
            frame = frame * w
        X = np.fft.rfft(frame, n=n_fft)
        stft[:, i] = X
    return stft

S_complex = stft_from_scratch(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window='hann')
S_power = (np.abs(S_complex) ** 2)  # power spectrogram

# --- mel filterbank (from linear freq bins) ---
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(m):
    return 700.0 * (10**(m / 2595.0) - 1.0)

def mel_filterbank(sr, n_fft, n_mels=128, fmin=0.0, fmax=None):
    if fmax is None:
        fmax = sr/2.
    # center freqs of fft bins
    fft_freqs = np.linspace(0.0, float(sr)/2.0, int(1 + n_fft//2))
    # mel scale
    mel_min = hz_to_mel(fmin)
    mel_max = hz_to_mel(fmax)
    mels = np.linspace(mel_min, mel_max, n_mels + 2)
    hz = mel_to_hz(mels)
    bins = np.floor((n_fft + 1) * hz / sr).astype(int)

    fb = np.zeros((n_mels, len(fft_freqs)))
    for m in range(1, n_mels + 1):
        f_m_minus = bins[m - 1]   # left
        f_m = bins[m]             # center
        f_m_plus = bins[m + 1]    # right

        if f_m_minus == f_m:
            f_m = f_m + 1
        for k in range(f_m_minus, f_m):
            fb[m-1, k] = (k - f_m_minus) / max(1, (f_m - f_m_minus))
        for k in range(f_m, f_m_plus):
            fb[m-1, k] = (f_m_plus - k) / max(1, (f_m_plus - f_m))
    # normalise
    enorm = 2.0 / (hz[2:n_mels+2] - hz[:n_mels])
    fb *= enorm[:, np.newaxis]
    return fb, fft_freqs

fb, fft_freqs = mel_filterbank(sr, n_fft, n_mels=n_mels, fmin=fmin, fmax=fmax)
# apply mel filterbank
mel_spec = np.dot(fb, S_power)   # shape (n_mels, t)
# convert to dB
mel_db = 10.0 * np.log10(np.maximum(1e-10, mel_spec))

# --- plot ---
plt.figure(figsize=(8, 4))
extent = [0, len(y)/sr, 0, n_mels]
plt.imshow(mel_db[::-1, :], aspect='auto', cmap='magma', extent=extent)
plt.xlabel('Time (s)')
plt.ylabel('Mel bins')
plt.title('Mel-spectrogram (from-scratch STFT + filterbank)')
plt.colorbar(format='%+2.0f dB')
plt.tight_layout()
plt.show()